# 01 — XLE Systematic Momentum: execution feasibility

> Esta es una evaluación exploratoria de ejecución para una candidata seleccionada en el mismo historial. No es una validación independiente de alpha ni una simulación operativa de IBKR.

## <b>Frozen specification</b>

La señal reproduce exactamente XLE Systematic Momentum de la notebook de origen: formación de 126 sesiones, OLS rolling de 252 sesiones con intercepto y MKT, SIZE y GV. La posición es long/cash sin apalancamiento. Buy & Hold, TSMOM total y Market Momentum se mantienen como referencias.

In [ ]:
from datetime import date
from pathlib import Path
import json

import numpy as np
import plotly.graph_objects as go
import polars as pl
from IPython.display import HTML, display

SYMBOLS = ("SPY", "XLE", "IWM", "IWD", "IWF")
RESEARCH_START = date(2018, 1, 2)
RESEARCH_END = date(2026, 11, 17)
OLS_WINDOW = 252
MOMENTUM_HORIZON = 126
INITIAL_CAPITAL = 10_000.0
SLIPPAGE_BPS = 5.0

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "data" / "gold" / "daily_series" / "etf").is_dir():
            return candidate
        project = candidate / "Momentum_systematic_strategies"
        if (project / "data" / "gold" / "daily_series" / "etf").is_dir():
            return project
    raise FileNotFoundError("No se encontró data/gold/daily_series/etf.")

PROJECT_ROOT = find_project_root(Path.cwd())
GOLD_ROOT = PROJECT_ROOT / "data" / "gold" / "daily_series" / "etf"
frames = {}
for symbol in SYMBOLS:
    path = GOLD_ROOT / f"{symbol}.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Gold no disponible para {symbol}: {path}")
    frames[symbol] = (
        pl.read_parquet(path)
        .with_columns(pl.col("session_date").cast(pl.Date))
        .filter(pl.col("session_date").is_between(RESEARCH_START, RESEARCH_END, closed="both"))
        .sort("session_date")
    )

xle_ohlc = frames["XLE"].select("session_date", "open", "close", "dividend_amount", "split_coefficient")
xle_ohlc.head()

## <b>Signal reproduction</b>

> Los precios ajustados se usan exclusivamente para retornos, atribución y señal. Los fills posteriores usan `open` sin ajustar. La verificación carga la notebook original en memoria y compara los pesos en fechas comunes.

In [ ]:
prices = frames["SPY"].select(pl.col("session_date").alias("date"), pl.col("adjusted_close").alias("SPY"))
for symbol in SYMBOLS[1:]:
    prices = prices.join(
        frames[symbol].select(pl.col("session_date").alias("date"), pl.col("adjusted_close").alias(symbol)),
        on="date",
        how="inner",
    )

returns = prices.sort("date").with_columns([pl.col(symbol).log().diff().alias(f"{symbol}_ret") for symbol in SYMBOLS]).drop_nulls()
model = returns.select("date", "XLE_ret", "SPY_ret", "IWM_ret", "IWF_ret", "IWD_ret").with_columns(
    pl.col("SPY_ret").alias("market"),
    (pl.col("IWM_ret") - pl.col("SPY_ret")).alias("size"),
    (pl.col("IWF_ret") - pl.col("IWD_ret")).alias("growth_value"),
)

dates = model.get_column("date").to_list()
factor_matrix = model.select("market", "size", "growth_value").to_numpy()
xle_returns = model.get_column("XLE_ret").to_numpy()
systematic_rows = []
for index in range(OLS_WINDOW, len(xle_returns)):
    design = np.column_stack((np.ones(OLS_WINDOW), factor_matrix[index - OLS_WINDOW:index]))
    coefficients = np.linalg.lstsq(design, xle_returns[index - OLS_WINDOW:index], rcond=None)[0]
    systematic_rows.append({
        "date": dates[index],
        "beta_asof_date": dates[index - 1],
        "actual_return": xle_returns[index],
        "market_return": factor_matrix[index, 0],
        "systematic_return": coefficients[0] + factor_matrix[index] @ coefficients[1:],
    })

attribution = pl.DataFrame(systematic_rows).with_columns((pl.col("actual_return") - pl.col("systematic_return")).alias("residual_return"))
signal_rows = []
for end in range(MOMENTUM_HORIZON - 1, attribution.height):
    window = attribution.slice(end - MOMENTUM_HORIZON + 1, MOMENTUM_HORIZON)
    signal_rows.append({
        "signal_date": window.get_column("date")[-1],
        "tsmom_weight": float(window.get_column("actual_return").sum() > 0),
        "market_weight": float(window.get_column("market_return").sum() > 0),
        "systematic_weight": float(window.get_column("systematic_return").sum() > 0),
    })

signals = pl.DataFrame(signal_rows)
return_dates = model.select(pl.col("date").alias("signal_date"), pl.col("date").shift(-1).alias("return_date"))
execution_schedule = signals.join(return_dates, on="signal_date", how="inner").join(
    xle_ohlc.rename({"session_date": "return_date"}), on="return_date", how="inner"
).join(
    xle_ohlc.select(pl.col("session_date").alias("signal_date"), pl.col("close").alias("signal_close")),
    on="signal_date", how="inner"
).drop_nulls().sort("signal_date")
execution_schedule.head()

In [ ]:
source_notebook = PROJECT_ROOT / "02_factor_momentum" / "notebooks" / "02_factor_momentum_signal_test.ipynb"
reference_notebook = json.loads(source_notebook.read_text(encoding="utf-8"))
reference_namespace = {"__name__": "__reference__"}
for cell in reference_notebook["cells"]:
    if cell["cell_type"] == "code":
        exec(compile("".join(cell["source"]), str(source_notebook), "exec"), reference_namespace)
    if cell.get("id") == "strategy-returns":
        break

reference_weights = reference_namespace["strategy_returns"].filter((pl.col("asset") == "XLE") & pl.col("alternative").is_in(["TSMOM total", "Market momentum", "Systematic momentum"])).select("signal_date", "return_date", "alternative", "weight")
candidate_weights = pl.concat([
    execution_schedule.select("signal_date", "return_date", pl.lit("TSMOM total").alias("alternative"), pl.col("tsmom_weight").alias("weight")),
    execution_schedule.select("signal_date", "return_date", pl.lit("Market momentum").alias("alternative"), pl.col("market_weight").alias("weight")),
    execution_schedule.select("signal_date", "return_date", pl.lit("Systematic momentum").alias("alternative"), pl.col("systematic_weight").alias("weight")),
])
signal_reproduction = candidate_weights.join(reference_weights, on=["signal_date", "return_date", "alternative"], how="inner", suffix="_reference")
signal_reproduction_audit = pl.DataFrame([{
    "common_rows": signal_reproduction.height,
    "max_abs_weight_difference": (signal_reproduction["weight"] - signal_reproduction["weight_reference"]).abs().max(),
}])
if signal_reproduction.height != reference_weights.height or signal_reproduction_audit["max_abs_weight_difference"][0] != 0:
    raise AssertionError("Las señales no reproducen la notebook de origen.")
signal_reproduction_audit

## <b>Order and accounting assumptions</b>

> Se reutilizan selectivamente del motor de rebalancing las ideas de orden preparada al cierre, fill adverso, compra financiable, cash/shares y ledger. No se reutiliza volatility targeting ni su lógica de rebalanceo.

Cada compra se dimensiona con el cierre sin ajustar conocido en `signal_date`; en la apertura siguiente se reduce si el fill real no es financiable. Los dividendos se acreditan como efectivo y los splits ajustan las acciones en la fecha registrada por Gold. Esto es una aproximación contable basada en la convención del proveedor.

In [ ]:
SCENARIOS = {
    "Open, no costs": {"commission": False, "slippage_bps": 0.0},
    "IBKR Pro Fixed": {"commission": True, "slippage_bps": 0.0},
    "IBKR Pro Fixed + 5 bps": {"commission": True, "slippage_bps": SLIPPAGE_BPS},
}
STRATEGIES = {
    "Buy & Hold": None,
    "TSMOM total": "tsmom_weight",
    "Market momentum": "market_weight",
    "Systematic momentum": "systematic_weight",
}

def ibkr_fixed_commission(shares, fill_price, enabled):
    if not enabled or shares == 0:
        return 0.0
    trade_value = abs(shares) * fill_price
    return min(max(abs(shares) * 0.005, 1.0), trade_value * 0.01)

def affordable_shares(cash, estimated_price, commission_enabled):
    quantity = int(cash // estimated_price)
    while quantity and quantity * estimated_price + ibkr_fixed_commission(quantity, estimated_price, commission_enabled) > cash:
        quantity -= 1
    return quantity

def run_execution(schedule, strategy, scenario):
    settings = SCENARIOS[scenario]
    shares, cash = 0, INITIAL_CAPITAL
    ledger, operations = [], []
    for row in schedule.iter_rows(named=True):
        split = row["split_coefficient"] or 1.0
        if split != 1.0:
            shares = int(round(shares * split))
        dividend_cash = shares * (row["dividend_amount"] or 0.0)
        cash += dividend_cash
        desired = 1.0 if strategy == "Buy & Hold" else row[STRATEGIES[strategy]]
        trade_shares, fill_price, commission, slippage_cost = 0, row["open"], 0.0, 0.0
        if desired == 0 and shares:
            trade_shares = -shares
        elif desired == 1 and shares == 0:
            estimated_fill = row["signal_close"] * (1 + settings["slippage_bps"] / 10_000)
            planned = affordable_shares(cash, estimated_fill, settings["commission"])
            actual_fill = row["open"] * (1 + settings["slippage_bps"] / 10_000)
            trade_shares = min(planned, affordable_shares(cash, actual_fill, settings["commission"]))
        if trade_shares:
            fill_price = row["open"] * (1 + np.sign(trade_shares) * settings["slippage_bps"] / 10_000)
            commission = ibkr_fixed_commission(trade_shares, fill_price, settings["commission"])
            slippage_cost = abs(trade_shares) * abs(fill_price - row["open"])
            if trade_shares > 0:
                cash -= trade_shares * fill_price + commission
            else:
                cash += abs(trade_shares) * fill_price - commission
            shares += trade_shares
            operations.append({
                "date": row["return_date"], "strategy": strategy, "scenario": scenario,
                "shares": trade_shares, "fill_price": fill_price, "open_price": row["open"],
                "notional": abs(trade_shares) * fill_price, "commission": commission, "slippage_cost": slippage_cost,
            })
        if cash < -1e-8:
            raise AssertionError("La simulación produjo efectivo negativo.")
        ledger.append({
            "date": row["return_date"], "signal_date": row["signal_date"], "strategy": strategy, "scenario": scenario,
            "desired_weight": desired, "shares": shares, "cash": cash, "close": row["close"],
            "nav": cash + shares * row["close"], "dividend_cash": dividend_cash,
        })
    return pl.DataFrame(ledger), pl.DataFrame(operations) if operations else pl.DataFrame(schema={"date": pl.Date, "strategy": pl.String, "scenario": pl.String, "shares": pl.Int64, "fill_price": pl.Float64, "open_price": pl.Float64, "notional": pl.Float64, "commission": pl.Float64, "slippage_cost": pl.Float64})


In [ ]:
ledger_frames, operation_frames = [], []
for scenario in SCENARIOS:
    for strategy in STRATEGIES:
        ledger, operations = run_execution(execution_schedule, strategy, scenario)
        ledger_frames.append(ledger)
        operation_frames.append(operations)

ledger = pl.concat(ledger_frames)
operations = pl.concat(operation_frames)
ledger.select("date", "strategy", "scenario", "shares", "cash", "nav").tail()

## <b>Friction scenarios</b>

La comisión usa IBKR Pro Fixed para acciones/ETFs estadounidenses: USD 0.005 por acción, mínimo USD 1 y tope 1% del valor negociado, consultado el 2026-11-19. El escenario de 5 bps es slippage adverso ilustrativo, no bid/ask histórico ni coste real medido de IBKR.

In [ ]:
summary_rows = []
for scenario in SCENARIOS:
    for strategy in STRATEGIES:
        history = ledger.filter((pl.col("scenario") == scenario) & (pl.col("strategy") == strategy)).sort("date")
        nav = history.get_column("nav").to_numpy()
        log_returns = np.diff(np.log(nav))
        drawdown = nav / np.maximum.accumulate(nav) - 1
        trades = operations.filter((pl.col("scenario") == scenario) & (pl.col("strategy") == strategy))
        summary_rows.append({
            "scenario": scenario, "strategy": strategy, "final_capital": nav[-1],
            "cagr_pct": ((nav[-1] / INITIAL_CAPITAL) ** (252 / len(log_returns)) - 1) * 100,
            "annualized_volatility_pct": log_returns.std(ddof=1) * np.sqrt(252) * 100,
            "sharpe": log_returns.mean() / log_returns.std(ddof=1) * np.sqrt(252),
            "max_drawdown_pct": drawdown.min() * 100, "operations": trades.height,
            "turnover_multiple": trades.get_column("notional").sum() / INITIAL_CAPITAL if trades.height else 0.0,
            "commission_total": trades.get_column("commission").sum() if trades.height else 0.0,
            "slippage_total": trades.get_column("slippage_cost").sum() if trades.height else 0.0,
            "time_invested_pct": (history.get_column("shares") > 0).mean() * 100,
        })

execution_summary = pl.DataFrame(summary_rows).with_columns(
    pl.all().exclude("scenario", "strategy", "operations").round(3)
)
execution_summary

In [ ]:
ideal_rows = []
for strategy in STRATEGIES:
    sample = reference_namespace["strategy_returns"].filter((pl.col("asset") == "XLE") & (pl.col("alternative") == strategy)).sort("return_date")
    returns = sample.get_column("strategy_return").to_numpy()
    wealth = np.exp(np.cumsum(returns))
    ideal_rows.append({
        "strategy": strategy,
        "idealized_return_pct": (np.exp(returns.mean() * 252) - 1) * 100,
        "idealized_sharpe": returns.mean() / returns.std(ddof=1) * np.sqrt(252),
        "idealized_max_drawdown_pct": (wealth / np.maximum.accumulate(wealth) - 1).min() * 100,
    })
idealized = pl.DataFrame(ideal_rows)
open_execution = execution_summary.filter(pl.col("scenario") == "Open, no costs").select(
    "strategy", pl.col("cagr_pct").alias("open_execution_cagr_pct"),
    pl.col("sharpe").alias("open_execution_sharpe"),
    pl.col("max_drawdown_pct").alias("open_execution_max_drawdown_pct"),
)
idealized_vs_execution = idealized.join(open_execution, on="strategy").with_columns(pl.all().exclude("strategy").round(3))
idealized_vs_execution

In [ ]:
figure = go.Figure()
for strategy in STRATEGIES:
    history = ledger.filter((pl.col("scenario") == "IBKR Pro Fixed + 5 bps") & (pl.col("strategy") == strategy)).sort("date")
    figure.add_scatter(x=history["date"], y=history["nav"], mode="lines", name=strategy)

figure.update_layout(title="XLE — capital after commissions and illustrative slippage", xaxis_title="date", yaxis_title="USD", template="plotly_white")
display(HTML(figure.to_html(include_plotlyjs="cdn", full_html=False)))

In [ ]:
systematic = ledger.filter((pl.col("scenario") == "IBKR Pro Fixed + 5 bps") & (pl.col("strategy") == "Systematic momentum")).sort("date")
tsmom = ledger.filter((pl.col("scenario") == "IBKR Pro Fixed + 5 bps") & (pl.col("strategy") == "TSMOM total")).sort("date")
cumulative_difference = (np.log(systematic["nav"].to_numpy() / INITIAL_CAPITAL) - np.log(tsmom["nav"].to_numpy() / INITIAL_CAPITAL)) * 100

figure = go.Figure()
figure.add_scatter(x=systematic["date"], y=cumulative_difference, mode="lines", name="Systematic − TSMOM")
figure.add_hline(y=0, line_dash="dash", line_color="gray")
figure.update_layout(title="XLE — cumulative execution differential", xaxis_title="date", yaxis_title="log capital difference (%)", template="plotly_white")
display(HTML(figure.to_html(include_plotlyjs="cdn", full_html=False)))

## <b>Interpretation</b>

- Open, no costs deja Systematic Momentum prácticamente alineado con la versión idealizada (15.632% frente a 15.571% CAGR); la diferencia procede de aperturas, acciones enteras, cash y la contabilidad explícita de dividendos.
- Con comisión IBKR Pro Fixed y 5 bps ilustrativos, Systematic Momentum cae de 15.632% a 15.054% CAGR y de USD 29,078 a USD 28,028 de capital final.
- Bajo esas fricciones, la señal sistemática conserva ventaja frente a TSMOM (3.931% CAGR) y Market Momentum (11.831%), con menor drawdown que ambos.
- Systematic Momentum registra 63 operaciones, cerca de 8.6 por año, una frecuencia manejable para un retail pero todavía sensible a fills y costes no observados.
- La contabilidad procesa 31 eventos de dividendos y un split desde Gold. Sus fechas son una aproximación del proveedor, no un calendario de pago o confirmación de broker.

## <b>Conclusion</b>

- La ejecución con aperturas, acciones enteras y cash no negativo reduce la rentabilidad sistemática sólo moderadamente bajo los supuestos usados; no elimina su ventaja histórica en XLE.
- La señal conserva ventaja frente a TSMOM y Market Momentum tras costes, mientras que Buy & Hold termina con menor capital y un drawdown mucho mayor.
- Las 63 operaciones históricas son operativamente razonables para un retail, pero los USD 68 de comisión y USD 456 de slippage ilustrativo no representan costes medidos de una cuenta IBKR.
- Bid/ask, profundidad, fills parciales, impuestos, dividendos pagados en fecha distinta, fees regulatorios y el sesgo de selección de XLE impiden tratar el resultado como PnL real o alpha confirmado.
- La candidata justifica una observación prospectiva en paper trading sólo con esta especificación congelada y con nuevas observaciones fuera del historial de selección.

Una observación prospectiva en paper trading sólo tendría sentido con esta señal congelada y con controles explícitos de órdenes, corporate actions y costes reales.